# Final preprocessing of the dataset (no blur)

For each selected image of `train` and `val` (**multiclass** version: bcc / mel / scc):

1. Crop the black borders of the dermatoscope
2. Central square + resize to 512x512
3. Color constancy (Shades of Gray, p=6, with optional brightness normalization)
4. Hair removal (cross-shaped black-hat + shape filter + inpainting, detected at 256 px)
5. Segmentation: Otsu on the S channel (directly on the image, no blur) + morphological cleaning

The alternatives tried at each step (and why these values were chosen) are in the exploration notebook.
The functions live in **`../preprocessing/`** (the same ones used by `Binary/02_preprocessing_dataset.ipynb`);
this notebook only sets the paths and the parameters and runs the whole dataset.

Reads the images from `dataset_multiclass/` (`train/{class}` and `val/{class}`, classes `bcc`, `mel`, `scc`) and saves to `preprocessed_multiclass/{split}/{class}/`:

- `<name>_preprocessed.png` (RGB 512x512, no blur, lossless PNG)
- `<name>_mask.png` (0/255, uint8, 512x512, aligned with the previous one)

plus a `preprocessed_multiclass/manifest.csv` with one row per image.

The process is **resumable**: if both files of an image already exist, it is not processed again.
The selection uses a fixed seed and is stratified by class and subtype, keeping the real proportions of the dataset.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import time
from collections import Counter
from pathlib import Path

# the preprocessing/ package lives in the repo folder, next to Binary/ and Multiclass/
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "preprocessing").is_dir())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO), os.environ.get("PYTHONPATH", "")])   # for the joblib workers

from preprocessing import PreprocParams, run_dataset, select_images, subtype, summarize_manifest, update_manifest

# ---------------- Configuration ----------------
ROOT = REPO                                       # folder with the data; change it if the data is elsewhere
DATA_DIR = ROOT / "dataset_multiclass"   # contains train/{bcc,mel,scc} and val/{bcc,mel,scc}
OUT_DIR = ROOT / "preprocessed_multiclass"        # the new images are saved here: {train,val}/{bcc,mel,scc}/
CLASSES = ["bcc", "mel", "scc"]

N_TRAIN = None            # None = all images
N_VAL = None              # None = all images
SEED = 42
OVERWRITE = False         # True = reprocess even if the output files already exist
N_JOBS = 4                # cores to use (-1 = all, -2 = all but one)

# Preprocessing parameters: the SAME in Binary and Multiclass (defaults in preprocessing/params.py)
P = PreprocParams(
    dark_thr=35,                  # 1. borders: gray < dark_thr = black
    min_area_frac=0.08,           # 1. borders: if the useful rectangle is smaller than this fraction, no crop is done
    target_size=512,              # 2. central square + resize
    sog_p=6,                      # 3. Shades of Gray
    normalize_brightness=True,    # 3. True = the estimated light is mapped to gray sog_target; False = only the tint is corrected
    sog_target=195.0,
    hair=dict(kernel_size=5,      # 4. hair (teammate's method): black-hat kernel width > hair thickness
              thresh=15,          #    minimum black-hat value to count as hair
              kernel_shape="cross",
              channel="gray",
              blur_ksize=3,       #    smooths the black-hat map to join broken hairs
              filter_shape=True,  #    discards round blobs (dots, lesion borders)
              min_elongation=3.0,
              min_area=20,
              max_fill=0.35,
              dilate_iter=1,
              inpaint_radius=5,
              inpaint_method="telea"),
    hair_ref_side=256,            # 4. hair is detected at this size (where the values were tuned) and inpainted at 512
    seg_channel="S",              # 5. channel for Otsu: "S", "L", "a", "b", "Cr", "gray"
)

## Stratified selection (class x subtype, real proportions)

In [2]:
DATA, OUT = Path(DATA_DIR), Path(OUT_DIR)


def class_dir(split, cls):
    """{DATA_DIR}/{split}/{cls}"""
    return DATA / split / cls


for sp in ["train", "val"]:
    for c in CLASSES:
        if not class_dir(sp, c).is_dir():
            raise FileNotFoundError(f"{class_dir(sp, c)} does not exist (check ROOT / DATA_DIR in the configuration)")
print("in:  ", DATA, "\nout: ", OUT)

jobs = select_images(class_dir, CLASSES, "train", N_TRAIN, SEED) + select_images(class_dir, CLASSES, "val", N_VAL, SEED)
print(f"\nTotal to process: {len(jobs)}")
for split in ["train", "val"]:
    cnt = Counter((c, subtype(f, c)) for s, c, f in jobs if s == split)
    print(f"  {split}: {dict(sorted(cnt.items()))}")

in:   C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\dataset_multiclass 
out:  C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\preprocessed_multiclass

Total to process: 6352
  train: {('bcc', 'bcc'): 1993, ('mel', 'mel'): 2713, ('scc', 'scc'): 376}
  val: {('bcc', 'bcc'): 498, ('mel', 'mel'): 678, ('scc', 'scc'): 94}


## Processing and saving (parallel and resumable)

In [3]:
t0 = time.time()
rows = run_dataset(jobs, OUT, P, overwrite=OVERWRITE, n_jobs=N_JOBS)
print(f"\nTotal time: {time.time() - t0:.0f}s  ({(time.time() - t0) / max(len(jobs), 1):.2f} s/img wall clock)")

  0%|          | 0/6352 [00:00<?, ?it/s]


Total time: 8s  (0.00 s/img wall clock)


## Manifest and summary

In [4]:
manifest_path = OUT / "manifest.csv"
new = update_manifest(rows, manifest_path)       # keeps full data from previous runs
print("manifest:", manifest_path, f"({len(new)} rows)\n")
summarize_manifest(new)

manifest: C:\Users\Lenovo\PycharmProjects\Melanoma-Challenge-ML-UdG\preprocessed_multiclass\manifest.csv (6352 rows)

status
ok    6352 

                n  seg_ok  area_frac_med  hair_frac_med
split class                                            
train bcc    1993   0.310          0.158          0.000
      mel    2713   0.371          0.241          0.001
      scc     376   0.279          0.196          0.001
val   bcc     498   0.321          0.161          0.000
      mel     678   0.350          0.242          0.001
      scc      94   0.266          0.189          0.001

Not cropped (fallback): 1
